
## 1. Load All Years Data
Load and inspect the combined all_years dataset.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Try to load existing cleaned data first; fallback to raw if not available
cleaned_dir = (Path.cwd().parent / "data" / "cleaned").resolve()
raw_dir = (Path.cwd().parent / "data" / "raw").resolve()

cleaned_path = cleaned_dir / "amazon_india_all_years_cleaned.csv"
raw_path = raw_dir / "amazon_india_all_years.csv"

if cleaned_path.exists():
    print(f"Loading existing cleaned data: {cleaned_path}")
    all_years_df = pd.read_csv(cleaned_path)
    print(f"Loaded shape: {all_years_df.shape}")
elif raw_path.exists():
    print(f"Cleaned file not found. Loading raw data: {raw_path}")
    all_years_df = pd.read_csv(raw_path)
    print(f"Loaded shape: {all_years_df.shape}")
else:
    raise FileNotFoundError(f"Neither cleaned ({cleaned_path}) nor raw ({raw_path}) file found.")

print(f"\nColumns in dataset: {list(all_years_df.columns)}")


Loading existing cleaned data: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
Loaded shape: (1127609, 34)

Columns in dataset: ['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']



## 2. Data Validation & Missing Values
Check data completeness and handle missing values.


In [2]:

# Check data completeness and missing values
print("Missing Values:")
print(all_years_df.isnull().sum())
print(f"\nDuplicate Rows: {all_years_df.duplicated().sum()}")
print(f"\nData Summary:")
print(all_years_df.info())


Missing Values:
transaction_id                 0
order_date                     0
customer_id                    0
product_id                     0
product_name                   0
category                       0
subcategory                    0
brand                          0
original_price_inr         36422
discount_percent               0
discounted_price_inr           0
quantity                       0
subtotal_inr                   0
delivery_charges           90201
final_amount_inr               0
customer_city                  0
customer_state                 0
customer_tier                  0
customer_spending_tier         0
customer_age_group        135315
payment_method                 0
delivery_days               6836
delivery_type                  0
is_prime_member                0
is_festival_sale               0
festival_name             777736
customer_rating           341696
return_status                  0
order_month                    0
order_year                 


## 3. Detect Price Outliers (Per Brand)
STEP 1: Statistical outliers using IQR within each brand.
STEP 2: Compute brand median and ratio to brand median.
STEP 3: Combine conditions – flag when statistical outlier AND exceeds 5× brand median.
STEP 4: Create tracking columns for audit trail.


In [3]:

# Define price columns
price_cols = ["original_price_inr", "discounted_price_inr"]

# Verify both columns exist
for col in price_cols:
    if col not in all_years_df.columns:
        raise KeyError(f"Expected column '{col}' not found.")

print(f"Price columns to analyze: {price_cols}\n")
print(f"Original Price Statistics:")
print(all_years_df["original_price_inr"].describe())
print(f"\nDiscounted Price Statistics:")
print(all_years_df["discounted_price_inr"].describe())


Price columns to analyze: ['original_price_inr', 'discounted_price_inr']

Original Price Statistics:
count    1.091187e+06
mean     8.318373e+04
std      4.114765e+05
min      1.067270e+03
25%      2.890339e+04
50%      4.662675e+04
75%      9.437859e+04
max      3.337169e+07
Name: original_price_inr, dtype: float64

Discounted Price Statistics:
count    1.127609e+06
mean     5.454134e+04
std      4.582480e+04
min      3.443300e+02
25%      2.278069e+04
50%      3.800119e+04
75%      7.410332e+04
max      4.207048e+05
Name: discounted_price_inr, dtype: float64


In [4]:

# STEP 1: Detect Statistical Outliers (Per Brand for both price columns)
def is_statistical_outlier(series):
    """Mark values outside IQR bounds as statistical outliers."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return (series < lower_bound) | (series > upper_bound)

# Apply to both price columns
for price_col in price_cols:
    col_name = f"is_stat_outlier_{price_col}"
    all_years_df[col_name] = all_years_df.groupby("brand")[price_col].transform(is_statistical_outlier)

print(f"Statistical outliers detected:")
for price_col in price_cols:
    col_name = f"is_stat_outlier_{price_col}"
    count = all_years_df[col_name].sum()
    pct = (count / len(all_years_df)) * 100
    print(f"  {price_col}: {count} ({pct:.2f}%)")


Statistical outliers detected:
  original_price_inr: 18198 (1.61%)
  discounted_price_inr: 12270 (1.09%)


In [5]:

# STEP 2: Compute Brand Medians and Ratios (for both price columns)
for price_col in price_cols:
    median_col = f"{price_col}_brand_median"
    ratio_col = f"ratio_to_{price_col}_median"
    
    all_years_df[median_col] = all_years_df.groupby("brand")[price_col].transform("median")
    all_years_df[ratio_col] = all_years_df[price_col] / all_years_df[median_col]

print(f"Brand medians computed for {all_years_df['brand'].nunique()} brands")
print(f"\nRatio statistics:")
for price_col in price_cols:
    ratio_col = f"ratio_to_{price_col}_median"
    print(f"\n{ratio_col}:")
    print(all_years_df[ratio_col].describe())


Brand medians computed for 29 brands

Ratio statistics:

ratio_to_original_price_inr_median:
count    1.091187e+06
mean     1.301397e+00
std      5.505442e+00
min      1.684132e-02
25%      7.193199e-01
50%      1.000000e+00
75%      1.299315e+00
max      4.589717e+02
Name: ratio_to_original_price_inr_median, dtype: float64

ratio_to_discounted_price_inr_median:
count    1.127609e+06
mean     1.072007e+00
std      5.732427e-01
min      6.482888e-03
25%      6.518932e-01
50%      1.000000e+00
75%      1.401642e+00
max      6.667431e+00
Name: ratio_to_discounted_price_inr_median, dtype: float64


In [6]:

# STEP 3: Combine Both Conditions (for both price columns)
# Correct ONLY when BOTH conditions are met:
# 1. Is a statistical outlier (per brand IQR)
# 2. Exceeds 5× brand median

SUSPICIOUS_THRESHOLD = 5.0

# Check each price column
for price_col in price_cols:
    ratio_col = f"ratio_to_{price_col}_median"
    threshold_col = f"ratio_exceeds_threshold_{price_col}"
    correction_col = f"needs_correction_{price_col}"
    stat_outlier_col = f"is_stat_outlier_{price_col}"
    
    all_years_df[threshold_col] = all_years_df[ratio_col] > SUSPICIOUS_THRESHOLD
    all_years_df[correction_col] = (
        all_years_df[stat_outlier_col] & 
        all_years_df[threshold_col]
    )

print(f"Outlier Detection Results:")
print(f"\n{price_cols[0]} (original_price_inr):")
print(f"  Statistical outliers: {all_years_df['is_stat_outlier_original_price_inr'].sum()}")
print(f"  Ratio exceeds {SUSPICIOUS_THRESHOLD}×: {all_years_df['ratio_exceeds_threshold_original_price_inr'].sum()}")
print(f"  Both conditions met (needs correction): {all_years_df['needs_correction_original_price_inr'].sum()}")

print(f"\n{price_cols[1]} (discounted_price_inr):")
print(f"  Statistical outliers: {all_years_df['is_stat_outlier_discounted_price_inr'].sum()}")
print(f"  Ratio exceeds {SUSPICIOUS_THRESHOLD}×: {all_years_df['ratio_exceeds_threshold_discounted_price_inr'].sum()}")
print(f"  Both conditions met (needs correction): {all_years_df['needs_correction_discounted_price_inr'].sum()}")


Outlier Detection Results:

original_price_inr (original_price_inr):
  Statistical outliers: 18198
  Ratio exceeds 5.0×: 5247
  Both conditions met (needs correction): 5247

discounted_price_inr (discounted_price_inr):
  Statistical outliers: 12270
  Ratio exceeds 5.0×: 1041
  Both conditions met (needs correction): 1041


In [7]:

# STEP 4: Create Tracking Columns (for both price columns)
for price_col in price_cols:
    median_col = f"{price_col}_brand_median"
    correction_col = f"needs_correction_{price_col}"
    corrected_col = f"{price_col}_corrected"
    status_col = f"correction_status_{price_col}"
    
    # Create corrected price column
    all_years_df[corrected_col] = all_years_df[price_col].copy()
    all_years_df.loc[all_years_df[correction_col], corrected_col] = (
        all_years_df.loc[all_years_df[correction_col], median_col]
    )
    
    # Create status column
    all_years_df[status_col] = "original"
    all_years_df.loc[all_years_df[correction_col], status_col] = "corrected_to_brand_median"

print(f"Tracking columns created:")
print(f"\noriginal_price_inr:")
print(f"  Original records: {(all_years_df['correction_status_original_price_inr'] == 'original').sum()}")
print(f"  Corrected records: {(all_years_df['correction_status_original_price_inr'] == 'corrected_to_brand_median').sum()}")

print(f"\ndiscounted_price_inr:")
print(f"  Original records: {(all_years_df['correction_status_discounted_price_inr'] == 'original').sum()}")
print(f"  Corrected records: {(all_years_df['correction_status_discounted_price_inr'] == 'corrected_to_brand_median').sum()}")

# Show samples if any corrections made
for price_col in price_cols:
    correction_col = f"needs_correction_{price_col}"
    if all_years_df[correction_col].any():
        print(f"\n\nSample of {price_col} records needing correction:")
        sample_cols = ["product_name", "brand", price_col, f"{price_col}_brand_median", f"ratio_to_{price_col}_median", f"correction_status_{price_col}"]
        display(all_years_df.loc[all_years_df[correction_col], sample_cols].head(10))


Tracking columns created:

original_price_inr:
  Original records: 1122362
  Corrected records: 5247

discounted_price_inr:
  Original records: 1126568
  Corrected records: 1041


Sample of original_price_inr records needing correction:


,product_name,brand,original_price_inr,original_price_inr_brand_median,ratio_to_original_price_inr_median,correction_status_original_price_inr
445,OnePlus Pad 4GB RAM Silver,OnePlus,3901009.0,65045.98,59.973099,corrected_to_brand_median
1194,Xiaomi Redmi Note 4G 64GB Black,Xiaomi,2631710.0,31752.20,82.882761,corrected_to_brand_median
1319,Xiaomi Redmi 2 16GB White,Xiaomi,4376740.0,31752.20,137.840528,corrected_to_brand_median
1478,Samsung OLED TV,Samsung,3888419.0,91130.98,42.668465,corrected_to_brand_median
1539,HP Inspiron 8GB RAM Silver,HP,723696.3,86451.97,8.371079,corrected_to_brand_median
1833,Lenovo Pad 8GB RAM Silver,Lenovo,6744491.0,79177.21,85.182226,corrected_to_brand_median
1897,LG QLED TV,LG,864775.3,105052.35,8.231851,corrected_to_brand_median
2161,Lenovo Tab M10 8GB RAM Black,Lenovo,563378.9,79177.21,7.115417,corrected_to_brand_median
2320,Xiaomi Redmi 2 32GB Black,Xiaomi,4420312.0,31752.20,139.212779,corrected_to_brand_median
2465,Samsung Galaxy S6 Edge 32GB Blue,Samsung,11728933.0,91130.98,128.704125,corrected_to_brand_median




Sample of discounted_price_inr records needing correction:


,product_name,brand,discounted_price_inr,discounted_price_inr_brand_median,ratio_to_discounted_price_inr_median,correction_status_discounted_price_inr
387248,Realme Slate 4GB RAM Silver,Realme,133911.47,24459.05,5.474925,corrected_to_brand_median
387624,Realme Mi Pad 4GB RAM Black,Realme,127101.92,24459.05,5.196519,corrected_to_brand_median
387927,Realme Mi Pad 4GB RAM Black,Realme,127101.92,24459.05,5.196519,corrected_to_brand_median
388803,Realme Galaxy Tab 8GB RAM Black,Realme,146234.56,24459.05,5.978751,corrected_to_brand_median
388959,Realme Slate 4GB RAM Silver,Realme,123413.60,24459.05,5.045723,corrected_to_brand_median
389191,Realme Galaxy Tab 8GB RAM Black,Realme,146234.56,24459.05,5.978751,corrected_to_brand_median
390182,Realme Galaxy Tab 8GB RAM Black,Realme,146234.56,24459.05,5.978751,corrected_to_brand_median
390271,Realme Galaxy Tab 8GB RAM Black,Realme,146234.56,24459.05,5.978751,corrected_to_brand_median
390577,Realme Galaxy Tab 8GB RAM Black,Realme,146234.56,24459.05,5.978751,corrected_to_brand_median
390666,Realme Mi Pad 4GB RAM Black,Realme,127101.92,24459.05,5.196519,corrected_to_brand_median



## 4. Outlier Summary by Year and Brand
Analyze outlier distribution across years and brands.


In [8]:

# Analyze by year (if year column exists)
year_col = [col for col in all_years_df.columns if 'year' in col.lower() and col != 'product_weight_kg']
if year_col:
    year_col = year_col[0]
    print(f"Statistical Outliers by {year_col}:")
    
    for price_col in price_cols:
        stat_col = f"is_stat_outlier_{price_col}"
        correction_col = f"needs_correction_{price_col}"
        print(f"\n{price_col}:")
        outliers_by_year = all_years_df.groupby(year_col).agg({
            stat_col: "sum",
            correction_col: "sum",
            price_col: "count"
        }).rename(columns={price_col: "total_records"})
        display(outliers_by_year)

# Analyze by brand - top 20
print(f"\n\nStatistical Outliers by Brand (top 20):")
for price_col in price_cols:
    stat_col = f"is_stat_outlier_{price_col}"
    correction_col = f"needs_correction_{price_col}"
    print(f"\n{price_col}:")
    outliers_by_brand = all_years_df.groupby("brand").agg({
        stat_col: "sum",
        correction_col: "sum",
        price_col: "count"
    }).rename(columns={price_col: "total_records"}).sort_values(stat_col, ascending=False)
    display(outliers_by_brand.head(20))

# Overall summary
print(f"\n\n=== OVERALL SUMMARY ===")
print(f"Total Records: {len(all_years_df)}")
print(f"Unique Brands: {all_years_df['brand'].nunique()}")

for price_col in price_cols:
    stat_col = f"is_stat_outlier_{price_col}"
    threshold_col = f"ratio_exceeds_threshold_{price_col}"
    correction_col = f"needs_correction_{price_col}"
    ratio_col = f"ratio_to_{price_col}_median"
    
    print(f"\n{price_col}:")
    print(f"  Statistical Outliers (IQR): {all_years_df[stat_col].sum()}")
    print(f"  Extreme Outliers (>5× median): {all_years_df[threshold_col].sum()}")
    print(f"  Records Needing Correction: {all_years_df[correction_col].sum()}")
    print(f"  Max Ratio to Brand Median: {all_years_df[ratio_col].max():.2f}×")


Statistical Outliers by order_year:

original_price_inr:


,is_stat_outlier_original_price_inr,needs_correction_original_price_inr,total_records
order_year,,,
2015,157,149,32094
2016,268,256,53508
2017,809,360,74920
2018,992,440,96262
2019,1764,536,117690
2020,3290,659,139073
2021,3533,774,133717
2022,2594,610,128377
2023,2229,549,123021



discounted_price_inr:


,is_stat_outlier_discounted_price_inr,needs_correction_discounted_price_inr,total_records
order_year,,,
2015,0,0,33165
2016,26,0,55275
2017,397,0,77385
2018,622,0,99495
2019,986,0,121605
2020,2931,289,143715
2021,2847,321,138187
2022,1594,127,132660
2023,1503,149,127132




Statistical Outliers by Brand (top 20):

original_price_inr:


,is_stat_outlier_original_price_inr,needs_correction_original_price_inr,total_records
brand,,,
Realme,4980,559,81872
Xiaomi,4736,746,151124
Apple,1514,487,108218
Samsung,1422,908,215525
Lenovo,813,108,22026
OnePlus,799,777,168728
MSI,789,50,9791
Audio-Technica,575,52,8401
Acer,435,72,13537



discounted_price_inr:


,is_stat_outlier_discounted_price_inr,needs_correction_discounted_price_inr,total_records
brand,,,
Realme,4176,928,84696
Xiaomi,3274,113,156154
Samsung,1085,0,222664
Apple,964,0,111827
Lenovo,697,0,22733
MSI,681,0,10122
LG,404,0,6410
Acer,336,0,13961
HP,202,0,9113




=== OVERALL SUMMARY ===
Total Records: 1127609
Unique Brands: 29

original_price_inr:
  Statistical Outliers (IQR): 18198
  Extreme Outliers (>5× median): 5247
  Records Needing Correction: 5247
  Max Ratio to Brand Median: 458.97×

discounted_price_inr:
  Statistical Outliers (IQR): 12270
  Extreme Outliers (>5× median): 1041
  Records Needing Correction: 1041
  Max Ratio to Brand Median: 6.67×



## 5. Export Cleaned Data with Outlier Flags
Save the dataset with outlier detection columns and summary tables.


In [9]:
# Save the cleaned data with all new outlier detection columns
output_path = cleaned_dir / "amazon_india_all_years_cleaned.csv"
all_years_df.to_csv(output_path, index=False)
print(f"✓ Saved consolidated cleaned data: {output_path}")
print(f"  Final dataset shape: {all_years_df.shape}")
print(f"\n  New outlier tracking columns added:")
outlier_cols = [col for col in all_years_df.columns if 'outlier' in col.lower() or 'correction' in col.lower() or 'ratio' in col.lower()]
for col in sorted(outlier_cols):
    print(f"    - {col}")


✓ Saved consolidated cleaned data: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
  Final dataset shape: (1127609, 48)

  New outlier tracking columns added:
    - correction_status_discounted_price_inr
    - correction_status_original_price_inr
    - is_stat_outlier_discounted_price_inr
    - is_stat_outlier_original_price_inr
    - needs_correction_discounted_price_inr
    - needs_correction_original_price_inr
    - ratio_exceeds_threshold_discounted_price_inr
    - ratio_exceeds_threshold_original_price_inr
    - ratio_to_discounted_price_inr_median
    - ratio_to_original_price_inr_median


In [11]:
# Display current shape of cleaned dataset
print(f"Current Cleaned Dataset Shape:")
print(f"  Rows: {all_years_df.shape[0]:,}")
print(f"  Columns: {all_years_df.shape[1]}")


Current Cleaned Dataset Shape:
  Rows: 1,127,609
  Columns: 48



# All Years Amazon India Outlier Detection
Detect and flag price outliers across all years (2015-2025) using statistical methods.


## Section 6: Duplicate Transaction Detection & Handling
Identify duplicates where same customer, product, date, and amount appear multiple times.
Distinguish between genuine duplicates (bulk orders) and data errors using multiple criteria.


In [12]:
# STEP 1: Identify Exact Duplicates
# Duplicates: same customer_id, product_id, order_date, original_price_inr

duplicate_cols = ['customer_id', 'product_id', 'order_date', 'original_price_inr']

# Mark duplicates (keep=False marks ALL occurrences as duplicate, not just the second one)
all_years_df['is_exact_duplicate'] = all_years_df.duplicated(subset=duplicate_cols, keep=False)

# Count total duplicates
total_duplicates = all_years_df['is_exact_duplicate'].sum()
unique_duplicate_sets = all_years_df[all_years_df['is_exact_duplicate']].drop_duplicates(subset=duplicate_cols).shape[0]

print(f"Duplicate Transaction Analysis:")
print(f"  Total duplicate records: {total_duplicates:,}")
print(f"  Unique duplicate sets: {unique_duplicate_sets:,}")
print(f"  Percentage of data: {(total_duplicates / len(all_years_df) * 100):.2f}%")


Duplicate Transaction Analysis:
  Total duplicate records: 11,042
  Unique duplicate sets: 5,521
  Percentage of data: 0.98%


In [15]:
def classify_duplicates_fast(df_dups):
    """
    Fast vectorized classification of duplicates.
    Check only key columns: quantity, delivery_charges, product_weight_kg, discounted_price_inr
    
    DATA_ERROR: All duplicates have identical key fields
    GENUINE_BULK_ORDER: Duplicates vary in quantity or weight or delivery charges
    """
    key_cols = ['quantity', 'delivery_charges', 'product_weight_kg', 'discounted_price_inr']
    
    # For each group, check if key columns are all identical
    def is_identical_group(group):
        return all(group[col].nunique() == 1 for col in key_cols if col in group.columns)
    
    result = pd.Series(index=df_dups.index, dtype='object', data='NOT_DUPLICATE')
    
    # Vectorized: assign based on whether all key fields are identical
    for dup_set in df_dups.groupby(['customer_id', 'product_id', 'order_date', 'original_price_inr']):
        group = dup_set[1]
        if is_identical_group(group):
            result[group.index] = 'DATA_ERROR'
        else:
            result[group.index] = 'GENUINE_BULK_ORDER'
    
    return result

# Faster classification
all_years_df['duplicate_type'] = 'NOT_DUPLICATE'
dup_indices = all_years_df[all_years_df['is_exact_duplicate']].index
all_years_df.loc[dup_indices, 'duplicate_type'] = classify_duplicates_fast(
    all_years_df.loc[dup_indices]
)

print(f"\nDuplicate Classification Results:")
dup_counts = all_years_df['duplicate_type'].value_counts()
for dup_type, count in dup_counts.items():
    pct = (count / len(all_years_df)) * 100
    print(f"  {dup_type}: {count:,} ({pct:.2f}%)")



Duplicate Classification Results:
  NOT_DUPLICATE: 1,116,909 (99.05%)
  DATA_ERROR: 9,844 (0.87%)
  GENUINE_BULK_ORDER: 856 (0.08%)


In [16]:
# STEP 3: Detailed Analysis of Duplicates by Category & Brand

# Analysis 1: Duplicates by Category
print("\nDuplicates by Category:")
dup_by_cat = all_years_df[all_years_df['is_exact_duplicate']].groupby('category').agg({
    'transaction_id': 'count',
    'duplicate_type': lambda x: (x == 'DATA_ERROR').sum()
}).rename(columns={'transaction_id': 'total_duplicates', 'duplicate_type': 'data_errors'})
dup_by_cat['genuine_orders'] = dup_by_cat['total_duplicates'] - dup_by_cat['data_errors']
dup_by_cat['error_pct'] = (dup_by_cat['data_errors'] / dup_by_cat['total_duplicates'] * 100).round(2)
print(dup_by_cat.sort_values('data_errors', ascending=False))

# Analysis 2: Duplicates by Brand (Top 15)
print("\n\nTop 15 Brands with Most Data Error Duplicates:")
dup_by_brand = all_years_df[all_years_df['duplicate_type'] == 'DATA_ERROR'].groupby('brand').agg({
    'transaction_id': 'count'
}).rename(columns={'transaction_id': 'error_count'}).sort_values('error_count', ascending=False).head(15)
print(dup_by_brand)

# Analysis 3: Sample of Data Errors
print("\n\nSample of Detected DATA ERRORS (Duplicate Transactions):")
error_samples = all_years_df[all_years_df['duplicate_type'] == 'DATA_ERROR'].drop_duplicates(
    subset=['customer_id', 'product_id', 'order_date', 'original_price_inr']
).head(10)
display(error_samples[['transaction_id', 'customer_id', 'product_name', 'brand', 'quantity', 
                       'order_date', 'original_price_inr', 'product_weight_kg']])

# Analysis 4: Sample of Genuine Bulk Orders
print("\n\nSample of GENUINE BULK ORDERS (Legitimate Duplicates):")
bulk_samples = all_years_df[all_years_df['duplicate_type'] == 'GENUINE_BULK_ORDER'].drop_duplicates(
    subset=['customer_id', 'product_id', 'order_date', 'original_price_inr']
).head(10)
display(bulk_samples[['transaction_id', 'customer_id', 'product_name', 'brand', 'quantity', 
                      'order_date', 'original_price_inr', 'product_weight_kg']])



Duplicates by Category:
             total_duplicates  data_errors  genuine_orders  error_pct
category                                                             
Electronics             11042         9844            1198      89.15


Top 15 Brands with Most Data Error Duplicates:
           error_count
brand                 
Samsung           1862
OnePlus           1546
Xiaomi            1396
Apple              998
Realme             736
Vivo               652
Oppo               544
Motorola           300
Lenovo             184
Alienware          130
Acer               116
ASUS               114
Fitbit             108
Noise              106
HP                 102


Sample of Detected DATA ERRORS (Duplicate Transactions):


,transaction_id,customer_id,product_name,brand,quantity,order_date,original_price_inr,product_weight_kg
814,TXN_2015_00000815,CUST_2015_00001928,OnePlus OnePlus X 32GB Black,OnePlus,1,2015-01-31,57182.39,0.20
941,TXN_2015_00000942,CUST_2015_00007653,Xiaomi Redmi 2 64GB White,Xiaomi,1,2015-01-03,35581.67,0.22
996,TXN_2015_00000997,CUST_2015_00003254,Samsung Galaxy S6 Edge 16GB White,Samsung,1,2015-01-28,97905.92,0.20
1577,TXN_2015_00001578,CUST_2015_00002994,OnePlus OnePlus X 32GB Blue,OnePlus,1,2015-01-24,75380.48,0.18
1626,TXN_2015_00001627,CUST_2015_00007323,Xiaomi Redmi 2 16GB Black,Xiaomi,1,2015-12-01,29534.70,0.20
1809,TXN_2015_00001810,CUST_2015_00005511,Xiaomi Redmi 2 16GB Black,Xiaomi,1,2015-01-31,29534.70,0.20
1851,TXN_2015_00001852,CUST_2015_00000037,OnePlus OnePlus 2 32GB Black,OnePlus,1,2015-01-23,102649.14,0.20
1999,TXN_2015_00002000,CUST_2015_00008125,OnePlus OnePlus X 16GB White,OnePlus,1,2015-01-01,94029.64,0.24
2137,TXN_2015_00002138,CUST_2015_00010624,Garmin Watch Deluxe,Garmin,2,2015-01-17,45363.47,0.08
2271,TXN_2015_00002272,CUST_2015_00000924,Xiaomi Redmi 2 32GB Blue,Xiaomi,2,2015-01-28,20175.12,0.19




Sample of GENUINE BULK ORDERS (Legitimate Duplicates):


,transaction_id,customer_id,product_name,brand,quantity,order_date,original_price_inr,product_weight_kg
1390,TXN_2015_00001391,CUST_2015_00005080,MSI Aspire 4GB RAM Black,MSI,1,2015-01-15,69584.33,2.66
2469,TXN_2015_00002470,CUST_2015_00010413,Noise Sports Watch Premium,Noise,1,2015-02-19,37013.54,0.03
4572,TXN_2015_00004573,CUST_2015_00005528,Dell MacBook 4GB RAM Silver,Dell,1,2015-03-01,29350.42,1.21
4931,TXN_2015_00004932,CUST_2015_00011164,Xiaomi Redmi Note 4G 16GB Black,Xiaomi,2,2015-03-26,13344.05,0.24
7998,TXN_2015_00007999,CUST_2015_00007720,Samsung Galaxy S6 Edge 32GB Blue,Samsung,2,2015-04-14,117289.33,0.16
10221,TXN_2015_00010222,CUST_2015_00009072,Apple iPhone 6 64GB Black,Apple,1,2015-05-07,118141.16,0.22
12247,TXN_2015_00012248,CUST_2015_00004003,Samsung Galaxy S6 16GB Black,Samsung,2,2015-06-17,123614.29,0.19
15707,TXN_2015_00015708,CUST_2015_00005973,Apple Pavilion 8GB RAM Silver,Apple,1,2015-07-16,51356.92,2.69
20625,TXN_2015_00020626,CUST_2015_00003851,Samsung Galaxy S6 32GB White,Samsung,1,2015-09-27,78047.80,0.21
25847,TXN_2015_00025848,CUST_2015_00011995,OnePlus OnePlus X 32GB White,OnePlus,1,2015-10-11,82953.19,0.19


In [18]:
# STEP 4: Duplicate Handling Strategy - VECTORIZED (Fast)

# Initialize all as KEEP
all_years_df['duplicate_action'] = 'KEEP'

# For DATA_ERROR duplicates: Keep first occurrence, mark rest for removal
data_errors = all_years_df[all_years_df['duplicate_type'] == 'DATA_ERROR'].copy()
if len(data_errors) > 0:
    # Add rank within each duplicate group (1st occurrence = rank 1)
    data_errors['dup_rank'] = data_errors.groupby(
        ['customer_id', 'product_id', 'order_date', 'original_price_inr'], 
        sort=False
    ).cumcount() + 1
    
    # Mark duplicates (keep rank 1, remove rank > 1)
    remove_indices = data_errors[data_errors['dup_rank'] > 1].index
    all_years_df.loc[remove_indices, 'duplicate_action'] = 'REMOVE_DUPLICATE'

# For GENUINE_BULK_ORDER: Keep all (vectorized assignment, no loop)
all_years_df.loc[all_years_df['duplicate_type'] == 'GENUINE_BULK_ORDER', 'duplicate_action'] = 'KEEP_BULK_ORDER'

# Summary
print("Duplicate Handling Strategy & Impact:")
print(f"\nAction Breakdown:")
action_counts = all_years_df['duplicate_action'].value_counts()
for action, count in action_counts.items():
    pct = (count / len(all_years_df)) * 100
    print(f"  {action}: {count:,} ({pct:.2f}%)")

records_to_remove = (all_years_df['duplicate_action'] == 'REMOVE_DUPLICATE').sum()
original_count = len(all_years_df)
cleaned_count = original_count - records_to_remove

print(f"\n✓ Deduplication Impact:")
print(f"  Original records: {original_count:,}")
print(f"  Records to remove: {records_to_remove:,}")
print(f"  Cleaned records: {cleaned_count:,}")
print(f"  Data reduction: {(records_to_remove / original_count * 100):.2f}%")

# Show examples of items being removed vs kept
print(f"\n\nExample: DATA ERROR Duplicates to Remove")
errors_to_remove = all_years_df[all_years_df['duplicate_action'] == 'REMOVE_DUPLICATE'].head(5)
display(errors_to_remove[['transaction_id', 'customer_id', 'product_name', 'quantity', 
                          'order_date', 'original_price_inr', 'duplicate_action']])


Duplicate Handling Strategy & Impact:

Action Breakdown:
  KEEP: 1,121,831 (99.49%)
  REMOVE_DUPLICATE: 4,922 (0.44%)
  KEEP_BULK_ORDER: 856 (0.08%)

✓ Deduplication Impact:
  Original records: 1,127,609
  Records to remove: 4,922
  Cleaned records: 1,122,687
  Data reduction: 0.44%


Example: DATA ERROR Duplicates to Remove


,transaction_id,customer_id,product_name,quantity,order_date,original_price_inr,duplicate_action
33000,TXN_2015_00016501_DUP,CUST_2015_00010086,Boat Earbuds Premium,1,2015-07-01,17174.29,REMOVE_DUPLICATE
33001,TXN_2015_00014828_DUP,CUST_2015_00003428,Xiaomi iPad 4GB RAM Black,1,2015-07-14,29675.17,REMOVE_DUPLICATE
33002,TXN_2015_00020641_DUP,CUST_2015_00011426,Samsung Galaxy S6 Edge 32GB White,1,2015-09-05,167952.46,REMOVE_DUPLICATE
33003,TXN_2015_00007274_DUP,CUST_2015_00004028,Xiaomi Mi 4i 64GB Black,1,2015-04-24,31102.54,REMOVE_DUPLICATE
33004,TXN_2015_00002272_DUP,CUST_2015_00000924,Xiaomi Redmi 2 32GB Blue,2,2015-01-28,20175.12,REMOVE_DUPLICATE


In [19]:
# STEP 5: Export Final Cleaned Dataset (Remove Data Errors, Keep Everything Else)

# Create final clean dataset: Remove only REMOVE_DUPLICATE rows
final_cleaned_df = all_years_df[all_years_df['duplicate_action'] != 'REMOVE_DUPLICATE'].copy()

print("="*70)
print("FINAL CLEANED DATASET EXPORT")
print("="*70)

print(f"\nOriginal dataset shape: {all_years_df.shape}")
print(f"Final cleaned shape: {final_cleaned_df.shape}")
print(f"Records removed: {len(all_years_df) - len(final_cleaned_df)}")

# Save final cleaned dataset
output_path = cleaned_dir / "amazon_india_all_years_cleaned.csv"
final_cleaned_df.to_csv(output_path, index=False)
print(f"\n✓ Saved final cleaned dataset: {output_path}")

# Summary of all cleaning operations
print(f"\n" + "="*70)
print("DATA CLEANING SUMMARY")
print("="*70)

print(f"\n1. PRICE OUTLIER DETECTION:")
print(f"   - Original price outliers (IQR): {all_years_df['is_stat_outlier_original_price_inr'].sum()}")
print(f"   - Discounted price outliers (IQR): {all_years_df['is_stat_outlier_discounted_price_inr'].sum()}")
print(f"   - Records needing price correction: {all_years_df['needs_correction_original_price_inr'].sum() + all_years_df['needs_correction_discounted_price_inr'].sum()}")

print(f"\n2. DUPLICATE DETECTION & REMOVAL:")
print(f"   - Total duplicate records found: {all_years_df['is_exact_duplicate'].sum()}")
print(f"   - Data error duplicates (removed): {(all_years_df['duplicate_action'] == 'REMOVE_DUPLICATE').sum()}")
print(f"   - Genuine bulk orders (kept): {(all_years_df['duplicate_action'] == 'KEEP_BULK_ORDER').sum()}")

print(f"\n3. FINAL DATASET:")
print(f"   - Original records: {len(all_years_df):,}")
print(f"   - Final clean records: {len(final_cleaned_df):,}")
print(f"   - Data retention rate: {(len(final_cleaned_df)/len(all_years_df)*100):.2f}%")
print(f"   - Total columns: {final_cleaned_df.shape[1]}")

# List all tracking columns added
tracking_cols = [col for col in final_cleaned_df.columns 
                 if any(x in col.lower() for x in ['outlier', 'correction', 'ratio', 'duplicate', 'action'])]
print(f"\n4. TRACKING COLUMNS ADDED ({len(tracking_cols)} columns):")
for col in sorted(tracking_cols):
    print(f"   - {col}")

print(f"\n" + "="*70)
print("✓ DATA CLEANING COMPLETE - READY FOR ANALYTICS")
print("="*70)


FINAL CLEANED DATASET EXPORT

Original dataset shape: (1127609, 51)
Final cleaned shape: (1122687, 51)
Records removed: 4922

✓ Saved final cleaned dataset: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv

DATA CLEANING SUMMARY

1. PRICE OUTLIER DETECTION:
   - Original price outliers (IQR): 18198
   - Discounted price outliers (IQR): 12270
   - Records needing price correction: 6288

2. DUPLICATE DETECTION & REMOVAL:
   - Total duplicate records found: 11042
   - Data error duplicates (removed): 4922
   - Genuine bulk orders (kept): 856

3. FINAL DATASET:
   - Original records: 1,127,609
   - Final clean records: 1,122,687
   - Data retention rate: 99.56%
   - Total columns: 51

4. TRACKING COLUMNS ADDED (14 columns):
   - correction_status_discounted_price_inr
   - correction_status_original_price_inr
   - duplicate_action
   - duplicate_type
   - is_exact_duplicate
   - is_stat_outlier_discounted_price_inr
   - is_stat_outlier_orig